In [1]:
import pickle
import pandas as pd
from pathlib import Path

from scipy.signal import find_peaks
import numpy as np

Dataset = 'HUST'

filename = f"processed_battery_features_{Dataset}.pkl"

with open(filename, "rb") as f:
    feature_df = pickle.load(f)

cell_ids = feature_df["cell_id"].unique().tolist()

print(f"{len(cell_ids)} cells found")
print(cell_ids[:10])

77 cells found
['HUST_1-1', 'HUST_1-2', 'HUST_1-3', 'HUST_1-4', 'HUST_1-5', 'HUST_1-6', 'HUST_1-7', 'HUST_1-8', 'HUST_10-1', 'HUST_10-2']


In [2]:
print(feature_df.shape)
feature_df.head()

(7886, 11)


,cell_id,cycle_number,SOH,I_mean,I_std,charge_duration,discharge_duration,V_mean,V_std,DTW_I,DTW_V
0,HUST_1-1,1,1.000994,-0.004839,2.731459,1927.0,2252.0,3.306669,0.209211,0.237899,1.512114
1,HUST_1-1,2,1.000000,-0.004372,2.754095,1866.0,2250.0,3.297230,0.199286,0.000000,0.000000
2,HUST_1-1,3,0.999747,-0.005082,2.753938,1862.0,2250.0,3.297379,0.200085,0.026142,0.304305
3,HUST_1-1,4,0.998969,-0.005504,2.753810,1862.0,2248.0,3.297319,0.200708,0.017825,0.591462
4,HUST_1-1,5,0.998469,-0.006003,2.753715,1863.0,2247.0,3.297430,0.201195,0.025613,0.950812


In [3]:
# Load the datasets for the selected cell IDs

data_folder = Path("Battery_life_Dataset/",Dataset)

datasets = []

for pkl_file in data_folder.glob("*.pkl"):

    with open(pkl_file, "rb") as f:
        data = pickle.load(f)

    if data["cell_id"] in cell_ids:
        datasets.append(data)

print(f"Loaded {len(datasets)} datasets")

Loaded 77 datasets


In [4]:
# Display the cell IDs of the loaded datasets

loaded_cell_ids = [d["cell_id"] for d in datasets]

print(sorted(loaded_cell_ids))

['HUST_1-1', 'HUST_1-2', 'HUST_1-3', 'HUST_1-4', 'HUST_1-5', 'HUST_1-6', 'HUST_1-7', 'HUST_1-8', 'HUST_10-1', 'HUST_10-2', 'HUST_10-3', 'HUST_10-4', 'HUST_10-5', 'HUST_10-6', 'HUST_10-7', 'HUST_10-8', 'HUST_2-2', 'HUST_2-3', 'HUST_2-4', 'HUST_2-5', 'HUST_2-6', 'HUST_2-7', 'HUST_2-8', 'HUST_3-1', 'HUST_3-2', 'HUST_3-3', 'HUST_3-4', 'HUST_3-5', 'HUST_3-6', 'HUST_3-7', 'HUST_3-8', 'HUST_4-1', 'HUST_4-2', 'HUST_4-3', 'HUST_4-4', 'HUST_4-5', 'HUST_4-6', 'HUST_4-7', 'HUST_4-8', 'HUST_5-1', 'HUST_5-2', 'HUST_5-3', 'HUST_5-4', 'HUST_5-5', 'HUST_5-6', 'HUST_5-7', 'HUST_6-1', 'HUST_6-2', 'HUST_6-3', 'HUST_6-4', 'HUST_6-5', 'HUST_6-6', 'HUST_6-8', 'HUST_7-1', 'HUST_7-2', 'HUST_7-3', 'HUST_7-4', 'HUST_7-5', 'HUST_7-6', 'HUST_7-7', 'HUST_7-8', 'HUST_8-1', 'HUST_8-2', 'HUST_8-3', 'HUST_8-4', 'HUST_8-5', 'HUST_8-6', 'HUST_8-7', 'HUST_8-8', 'HUST_9-1', 'HUST_9-2', 'HUST_9-3', 'HUST_9-4', 'HUST_9-5', 'HUST_9-6', 'HUST_9-7', 'HUST_9-8']


In [5]:
# Feature-Engineering: Berechnung von SOH, Strom- und Spannungsstatistiken sowie Lade-/Entladezeiten

v_shape_features = []

for data in datasets:
    ref_v = np.array(
        data["cycle_data"][1]["voltage_in_V"],
        dtype=np.float32
    )

    for cycle in data["cycle_data"]:

        v = np.array(cycle["voltage_in_V"], dtype=np.float32)

        dv = np.diff(v)
        d2v = np.diff(dv)

        peaks, _ = find_peaks(v)

        # gleiche Länge erzwingen
        min_len = min(len(ref_v), len(v))

        ref = ref_v[:min_len]
        x = v[:min_len]

        # einfache Distanzmetriken
        rmse_v = np.sqrt(np.mean((ref - x) ** 2))

        area_diff_v = np.mean(np.abs(ref - x))

        corr_v = np.corrcoef(ref, x)[0, 1]

        max_dev_v = np.max(np.abs(ref - x))

        # Ableitungen vergleichen
        dref = np.diff(ref)
        dx = np.diff(x)

        min_len_diff = min(len(dref), len(dx))

        slope_rmse_v = np.sqrt(
            np.mean(
                (dref[:min_len_diff] - dx[:min_len_diff]) ** 2
            )
        )

        v_shape_features.append({
            "cell_id": data["cell_id"],
            "cycle_number": cycle["cycle_number"],

            # bisherige Features
            "V_range": np.max(v) - np.min(v),
            "V_slope_mean": np.mean(np.abs(dv)),
            "V_curvature": np.mean(np.abs(d2v)),
            "V_n_peaks": len(peaks),

            # neue DTW-Alternativen
            "RMSE_V": rmse_v,
            "AreaDiff_V": area_diff_v,
            "Corr_V": corr_v,
            "MaxDev_V": max_dev_v,
            "SlopeRMSE_V": slope_rmse_v
        })

v_shape_df = pd.DataFrame(v_shape_features)

In [6]:
# Merge the new features with the existing feature DataFrame

feature_df = feature_df.merge(
    v_shape_df,
    on=["cell_id", "cycle_number"],
    how="left"
)

In [7]:
# Display the columns of the merged DataFrame and a sample of the new features

print(feature_df.columns.tolist())
print(feature_df[
    [
        "V_range",
        "V_slope_mean",
        "V_curvature",
        "V_n_peaks"
    ]
].head())

['cell_id', 'cycle_number', 'SOH', 'I_mean', 'I_std', 'charge_duration', 'discharge_duration', 'V_mean', 'V_std', 'DTW_I', 'DTW_V', 'V_range', 'V_slope_mean', 'V_curvature', 'V_n_peaks', 'RMSE_V', 'AreaDiff_V', 'Corr_V', 'MaxDev_V', 'SlopeRMSE_V']
   V_range  V_slope_mean  V_curvature  V_n_peaks
0   1.6518      0.003793     0.002341         16
1   1.6589      0.003912     0.002143         20
2   1.6701      0.003923     0.002092         20
3   1.6546      0.003908     0.002045         23
4   1.6183      0.003853     0.002269         16


In [8]:
exportfilename = f"processed_battery_features_v2_{Dataset}.pkl"

with open(exportfilename, "wb") as f:
    pickle.dump(feature_df, f)